# Spectral Detectors vs the Matching Attack

**Paper role: Section 5 (does the attack actually evade a detector, and what survives?).**

We train two detectors and push matched fakes through them:

1. **Radial detector** - logistic regression on the log radial magnitude
   profile (the fingerprint from Notebook 1). The attack is designed to beat this.
2. **Residual detector** - logistic regression on the *spectral residual*
   `R = |FFT| / ring-mean(|FFT|)`, which exposes 2-D peaks (upsampling /
   checkerboard artifacts) that live *inside* a frequency ring. Radial matching
   scales each ring uniformly, so it cannot remove these.

Expected result: radial evasion ~1.0 (fully fooled), residual evasion ~0.0
(still caught). That is the paper's punchline: spectral-*magnitude* detectors
are fragile, but phase / 2-D-structure detectors are not.

This notebook runs **CPU-only** on synthetic data. To use real images set
`USE_SYNTHETIC = False` (generating fakes needs a GPU - see `MATH.md`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- inline helpers (standalone; mirror src/spectral.py + src/detector.py) ----
def _radius_map(h, w):
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    return np.round(np.sqrt((y - cy) ** 2 + (x - cx) ** 2)).astype(int)

def radial_profile(ch):
    F = np.fft.fftshift(np.fft.fft2(ch)); mag = np.abs(F)
    r = _radius_map(*ch.shape); mr = min(ch.shape) // 2
    t = np.bincount(r.ravel(), weights=mag.ravel()); c = np.bincount(r.ravel())
    return t[:mr] / np.maximum(c[:mr], 1)

def _match_channel(ch, target, gain_clip=(0.1, 12.0), smooth=3, preserve_dc=True):
    F = np.fft.fftshift(np.fft.fft2(ch)); r = _radius_map(*ch.shape); mr = len(target)
    gain = target / (radial_profile(ch) + 1e-12)
    if smooth > 1: gain = np.convolve(gain, np.ones(smooth) / smooth, mode='same')
    gain = np.clip(gain, *gain_clip)
    if preserve_dc: gain[0] = 1.0
    g = gain[np.clip(r, 0, mr - 1)]; g[r >= mr] = 1.0
    if preserve_dc: g[r == 0] = 1.0
    return np.fft.ifft2(np.fft.ifftshift(F * g)).real

def spectral_match(img, target):
    out = _match_channel(img, target)
    return np.clip(out, 0, 255)

def radial_features(ch):
    return np.log(radial_profile(ch) + 1e-8)

def _kurt(v):
    mu, sd = v.mean(), v.std() + 1e-12
    return ((v - mu) ** 4).mean() / sd ** 4

def residual_features(ch):
    prof = radial_profile(ch)
    A = np.abs(np.fft.fftshift(np.fft.fft2(ch))); r = _radius_map(*ch.shape)
    R = A / (prof[np.clip(r, 0, len(prof) - 1)] + 1e-8); v = R[r >= 20]
    return np.array([np.percentile(v, 90), np.percentile(v, 95), np.percentile(v, 99),
                     v.max(), _kurt(v), v.mean()])

class LogReg:
    def __init__(s, lr=0.5, epochs=2000, l2=1e-3): s.lr, s.epochs, s.l2 = lr, epochs, l2
    def fit(s, X, y):
        y = np.asarray(y, float); s.mu = X.mean(0); s.sd = X.std(0) + 1e-8
        Xs = (X - s.mu) / s.sd; n, d = Xs.shape; s.w = np.zeros(d); s.b = 0.0
        for _ in range(s.epochs):
            p = 1 / (1 + np.exp(-(Xs @ s.w + s.b)))
            s.w -= s.lr * (Xs.T @ (p - y) / n + s.l2 * s.w); s.b -= s.lr * (p - y).mean()
        return s
    def predict(s, X): return (1 / (1 + np.exp(-(((X - s.mu) / s.sd) @ s.w + s.b))) >= .5).astype(int)

def build(imgs, kind):
    fn = radial_features if kind == 'radial' else residual_features
    return np.stack([fn(im) for im in imgs])

def run_evasion(reals, fakes, matched, kind, seed=0):
    rng = np.random.default_rng(seed)
    Xr, Xf, Xm = build(reals, kind), build(fakes, kind), build(matched, kind)
    nr, nf = len(Xr), len(Xf); ri, fi = rng.permutation(nr), rng.permutation(nf)
    rtr, rte, ftr, fte = ri[:nr//2], ri[nr//2:], fi[:nf//2], fi[nf//2:]
    Xtr = np.vstack([Xr[rtr], Xf[ftr]]); ytr = np.r_[np.zeros(len(rtr)), np.ones(len(ftr))]
    Xte = np.vstack([Xr[rte], Xf[fte]]); yte = np.r_[np.zeros(len(rte)), np.ones(len(fte))]
    d = LogReg().fit(Xtr, ytr)
    return {'clean_acc': (d.predict(Xte) == yte).mean(),
            'fake_recall': (d.predict(Xf[fte]) == 1).mean(),
            'evasion': (d.predict(Xm[fte]) == 0).mean()}
print('helpers ready')

## Data: synthetic (CPU) or real (needs GPU to make fakes)

In [ ]:
USE_SYNTHETIC = True   # <-- set False to plug in real CIFAR + SD-generated fakes
N, K = 256, 60

if USE_SYNTHETIC:
    cy = cx = N // 2; yy, xx = np.ogrid[:N, :N]
    rad = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    def make(seed, hf=1.0, grid=False):
        g = np.random.default_rng(seed).standard_normal((N, N))
        F = np.fft.fftshift(np.fft.fft2(g)) * (1 / (rad + 1)) * np.where(rad >= 60, hf, 1.0)
        out = np.fft.ifft2(np.fft.ifftshift(F)).real
        out -= out.min(); out /= out.max(); out *= 255
        if grid:
            m = np.arange(N)
            out = out + 18 * np.cos(2*np.pi*70*m[:, None]/N) + 18 * np.cos(2*np.pi*70*m[None, :]/N)
            out = np.clip(out, 0, 255)
        return out
    reals = [make(s) for s in range(K)]
    fakes = [make(1000 + s, hf=1/20, grid=True) for s in range(K)]
else:
    # from PIL import Image
    # reals = [np.asarray(Image.open(p).convert('L').resize((N,N)),float) for p in REAL_PATHS]
    # fakes = [np.asarray(Image.open(p).convert('L').resize((N,N)),float) for p in FAKE_PATHS]
    raise NotImplementedError('Provide real + fake image lists (fakes need a GPU to generate).')

target = np.mean([radial_profile(im) for im in reals], axis=0)
matched = [spectral_match(im, target) for im in fakes]
print(f'{K} real / {K} fake / {K} matched ready')

## Run the evasion experiment

In [ ]:
rows = {k: run_evasion(reals, fakes, matched, k) for k in ('radial', 'residual')}
print(f"{'detector':>10}{'clean acc':>12}{'fake recall':>13}{'evasion':>10}")
for k, v in rows.items():
    print(f"{k:>10}{v['clean_acc']:>12.2f}{v['fake_recall']:>13.2f}{v['evasion']:>10.2f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['radial', 'residual'], [rows['radial']['evasion'], rows['residual']['evasion']],
       color=['red', 'green'])
ax.set_ylabel('evasion rate (matched fakes called real)'); ax.set_ylim(0, 1)
ax.set_title('Spectral matching defeats the radial detector, not the residual one')
for i, k in enumerate(('radial', 'residual')):
    ax.text(i, rows[k]['evasion'] + 0.02, f"{rows[k]['evasion']:.2f}", ha='center')
plt.tight_layout(); plt.savefig('/content/fig5_evasion.png', dpi=150, bbox_inches='tight'); plt.show()

## Interpretation (draft text for Section 5)

- The radial detector has ~perfect clean accuracy yet ~1.0 evasion: the spectral
  fingerprint it relies on is exactly what the attack overwrites.
- The residual detector keeps ~0.0 evasion: 2-D peaks survive radial matching,
  so they remain a usable cue.
- Takeaway: report frequency detectors as **necessary-but-not-sufficient**;
  robust detection needs phase / 2-D-structure / learned features.